In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
#import 
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
from helpers import merge_bkg, merge_cutflows 

In [3]:
vr = "24"
output = coffea.util.load(f"outputs/bkg_{vr}_all.coffea")
sig2mu = [
    "2Mu2E_200GeV_5p0GeV_100p0mm", #changing MXX
    "2Mu2E_500GeV_5p0GeV_80p0mm",
    "2Mu2E_800GeV_5p0GeV_50p0mm",
    "2Mu2E_1000GeV_5p0GeV_40p0mm",
    
    "2Mu2E_500GeV_0p25GeV_4p0mm", #changing MZd
    "2Mu2E_500GeV_1p2GeV_19p0mm",

    "2Mu2E_500GeV_1p2GeV_0p019mm", #changing Lxy
    "2Mu2E_500GeV_1p2GeV_0p19mm",
    "2Mu2E_500GeV_1p2GeV_1p9mm",
    "2Mu2E_500GeV_1p2GeV_9p6mm",
    "2Mu2E_500GeV_1p2GeV_19p0mm", 
]

sig4mu = [
    "4Mu_200GeV_5p0GeV_200p0mm",
    "4Mu_500GeV_5p0GeV_80p0mm",
    "4Mu_800GeV_5p0GeV_50p0mm",
    "4Mu_1000GeV_5p0GeV_40p0mm",

    "4Mu_500GeV_0p25GeV_0p004mm",
    "4Mu_500GeV_1p2GeV_19p0mm",

    "4Mu_500GeV_1p2GeV_0p019mm",
    "4Mu_500GeV_1p2GeV_0p19mm",
    "4Mu_500GeV_1p2GeV_1p9mm",
    "4Mu_500GeV_1p2GeV_9p6mm",
    "4Mu_500GeV_1p2GeV_19p0mm",
]

bkgdyj = ["DYJetsToMuMu_M10to50", "DYJetsToMuMu_M50",]

bkgttj = ["TTJets"]

bkgqcd = ["QCD_Pt15To20",    "QCD_Pt20To30",    "QCD_Pt30To50",    "QCD_Pt50To80",    "QCD_Pt80To120",
          "QCD_Pt120To170",    "QCD_Pt170To300",    "QCD_Pt300To470",    "QCD_Pt470To600",    "QCD_Pt600To800",
    # "QCD_Pt800To1000", 
    "QCD_Pt1000",       
]

# bkgalq = bkgqcd + bkgocd
pros = [sig2mu, sig4mu, bkgdyj, bkgttj, bkgqcd]

mxx2 = sig2mu[:4]
mzd2 = sig2mu[4:7]
lxy2 = sig2mu[7:]

mxx4 = sig4mu[:4]
mzd4 = sig4mu[4:7]
lxy4 = sig4mu[7:]

#channels: cuts to be applied (slections.yaml) 
channels = ["baseNoLj", 
            "bkg_iso",
            "bkg_iso_disp",
            "bkg_iso_disp_2lj",
            "bkg_iso_disp_2lj_2mu2e",
            "bkg_iso_disp_2lj_4mu",
            "bkg_iso_disp_2lj_2mu2e_dphi",
            "bkg_iso_disp_2lj_4mu_dphi",
           ]

ch1 = channels[0] #base
ch2 = channels[1] # iso
ch3 = channels[2] # iso + disp
ch4 = channels[3] # iso + disp + 2lj
ch5 = channels[4] # iso + disp + 2lj + 2mu2e
ch6 = channels[5] # iso + disp + 2lj + 4mu
ch7 = channels[6] # iso + disp + 2lj + 2mu2e + dphi
ch8 = channels[7] # iso + disp + 2lj + 4mu + dphi

In [2]:
vr = "24a"
output = coffea.util.load(f"outputs/bkg_{vr}.coffea")

tmulxy1 = [
    "2Mu2E_500GeV_0p25GeV_0p004mm", "2Mu2E_500GeV_0p25GeV_0p04mm", "2Mu2E_500GeV_0p25GeV_0p4mm", "2Mu2E_500GeV_0p25GeV_2p0mm", "2Mu2E_500GeV_0p25GeV_4p0mm"]

bkgdyj = ["DYJetsToMuMu_M10to50", "DYJetsToMuMu_M50",]

bkgttj = ["TTJets"]

bkgqcd = ["QCD_Pt15To20",    "QCD_Pt20To30",    "QCD_Pt30To50",    "QCD_Pt50To80",    "QCD_Pt80To120",
          "QCD_Pt120To170",    "QCD_Pt170To300",    "QCD_Pt300To470",    "QCD_Pt470To600",    "QCD_Pt600To800",
    # "QCD_Pt800To1000", 
    "QCD_Pt1000",       
]

channels = ["baseNoLj", 
            "bkg_iso",
            "bkg_iso_disp",
            "bkg_iso_disp_2lj",
            "bkg_iso_disp_2lj_2mu2e",
            "bkg_iso_disp_2lj_4mu",
            "bkg_iso_disp_2lj_2mu2e_dphi",
            "bkg_iso_disp_2lj_4mu_dphi",
           ]

ch1 = channels[0] #base
ch2 = channels[1] # iso
ch3 = channels[2] # iso + disp
ch4 = channels[3] # iso + disp + 2lj
ch5 = channels[4] # iso + disp + 2lj + 2mu2e
ch6 = channels[5] # iso + disp + 2lj + 4mu
ch7 = channels[6] 

In [3]:
# print(sig2mu[6:])
for i in tmulxy1:
    print(i)
    output[i]["cutflow"][ch7].print_table()
    print()

2Mu2E_500GeV_0p25GeV_0p004mm
cut name          raw N    weighted N    weighted %
--------------  -------  ------------  ------------
None            10991.0          59.8         100.0
pass triggers    6546.0          35.6          59.6
PV filter        6546.0          35.6          59.6
>=2 LJs          6209.0          33.8          56.5
2mu2e            6010.0          32.7          54.7
LJ-LJ dPhi > 2   5958.0          32.4          54.2

2Mu2E_500GeV_0p25GeV_0p04mm
cut name          raw N    weighted N    weighted %
--------------  -------  ------------  ------------
None            13091.0          59.8         100.0
pass triggers    7689.0          35.1          58.7
PV filter        7689.0          35.1          58.7
>=2 LJs          7110.0          32.5          54.3
2mu2e            6773.0          31.0          51.7
LJ-LJ dPhi > 2   6708.0          30.7          51.2

2Mu2E_500GeV_0p25GeV_0p4mm
cut name          raw N    weighted N    weighted %
--------------  -------  -----

In [22]:
DYJ = merge_cutflows(output, bkgdyj, ch7)
TTJ = merge_cutflows(output, bkgttj, ch7)
QCD = merge_cutflows(output, bkgqcd + bkgocd, ch7)

In [23]:
backgrounds = {
    "DYJ": bkgdyj,
    "TTJ": bkgttj,
    "QCD": bkgqcd + bkgocd,
}

merged_cf = {}

for name, samples in backgrounds.items():
    merged_cf[name] = merge_cutflows(output, samples, ch7)

In [28]:
for name, cf in merged_cf.items():
    print(f"\n{name}")
    print(f"{'cut name':<20} {'raw':>12} {'weighted':>15}")
    print("-" * 50)

    for cut, vals in cf.items():
        print(
            f"{cut:<20}"
            f"{vals['raw']:>12,.0f}"
            f"{vals['weighted']:>15,.1f}"
        )


DYJ
cut name                      raw        weighted
--------------------------------------------------
None                   1,858,145   75,826,240.0
pass triggers            925,670   36,425,488.0
PV filter                925,670   36,425,488.0
>=2 LJs                  739,173   28,394,172.0
2mu2e                         19        2,197.8
LJ-LJ dPhi > 2                14        1,982.2

TTJ
cut name                      raw        weighted
--------------------------------------------------
None                   1,036,173    7,751,379.0
pass triggers             63,796      477,287.1
PV filter                 63,777      477,197.3
>=2 LJs                   36,143      270,432.0
2mu2e                        497        3,718.7
LJ-LJ dPhi > 2               292        2,184.8

QCD
cut name                      raw        weighted
--------------------------------------------------
None                   1,154,161  499,645,664.0
pass triggers             46,128   32,719,034.0
PV filter 

In [26]:
all_bkgs = bkgdyj + bkgttj + bkgqcd + bkgocd
total_bkg = merge_cutflows(output, all_bkgs, ch7)

In [27]:
for cut, vals in total_bkg.items():
    print(
        f"{cut:<20}"
        f"{vals['raw']:>12,.0f}"
        f"{vals['weighted']:>15,.1f}"
    )

None                   4,048,479  583,223,360.0
pass triggers          1,035,594   69,621,816.0
PV filter              1,035,575   69,621,720.0
>=2 LJs                  785,258   30,353,402.0
2mu2e                      1,008       90,491.6
LJ-LJ dPhi > 2               598       76,798.7


In [21]:
dyj_cf = merge_cutflows(output, bkgdyj, ch7)

print(f"{'cut name':<20} {'raw N':>12} {'weighted N':>15}")
print("-"*50)

for cut, vals in dyj_cf.items():
    print(
        f"{cut:<20}"
        f"{vals['raw']:>12,.0f}"
        f"{vals['weighted']:>15,.1f}"
    )

cut name                    raw N      weighted N
--------------------------------------------------
None                   1,036,173    7,751,379.0
pass triggers             63,796      477,287.1
PV filter                 63,777      477,197.3
>=2 LJs                   36,143      270,432.0
2mu2e                        497        3,718.7
LJ-LJ dPhi > 2               292        2,184.8


In [19]:
dyj_cf = merge_cutflows(output, bkgdyj, ch7)

print(f"{'cut name':<20} {'raw N':>12} {'weighted N':>15}")
print("-"*50)

for cut, vals in dyj_cf.items():
    print(
        f"{cut:<20}"
        f"{vals['raw']:>12,.0f}"
        f"{vals['weighted']:>15,.1f}"
    )

cut name                    raw N      weighted N
--------------------------------------------------
None                   1,858,145   75,826,240.0
pass triggers            925,670   36,425,488.0
PV filter                925,670   36,425,488.0
>=2 LJs                  739,173   28,394,172.0
2mu2e                         19        2,197.8
LJ-LJ dPhi > 2                14        1,982.2


In [8]:
print("\n TTJ \n")
output[bkgttj[0]]["cutflow"][ch7].print_table()


 TTJ 

cut name            raw N    weighted N    weighted %
--------------  ---------  ------------  ------------
None            1036173.0     7751379.0         100.0
pass triggers     63796.0      477287.1           6.2
PV filter         63777.0      477197.3           6.2
>=2 LJs           36143.0      270432.0           3.5
2mu2e               497.0        3718.7           0.0
LJ-LJ dPhi > 2      292.0        2184.8           0.0


In [10]:
for i in bkgdyj:
    print(i)
    output[i]["cutflow"][ch7].print_table()


DYJetsToMuMu_M10to50
cut name          raw N    weighted N    weighted %
--------------  -------  ------------  ------------
None            11163.0     5532717.0         100.0
pass triggers    1443.0      687547.6          12.4
PV filter        1443.0      687547.6          12.4
>=2 LJs           144.0       72373.8           1.3
2mu2e               3.0        1507.8           0.0
LJ-LJ dPhi > 2      3.0        1507.8           0.0
DYJetsToMuMu_M50
cut name            raw N    weighted N    weighted %
--------------  ---------  ------------  ------------
None            1846982.0    70293520.0         100.0
pass triggers    924227.0    35737940.0          50.8
PV filter        924227.0    35737940.0          50.8
>=2 LJs          739029.0    28321798.0          40.3
2mu2e                16.0         690.0           0.0
LJ-LJ dPhi > 2       11.0         474.4           0.0


In [11]:
for i in bkgqcd:
    print(i)
    output[i]["cutflow"][ch5].print_table()
    print("\n")

for i in bkgocd:
    print(i)
    output[i]["cutflow"][ch5].print_table()
    print("\n")

QCD_Pt15To20
cut name         raw N    weighted N    weighted %
-------------  -------  ------------  ------------
None             142.0    22589410.0         100.0
pass triggers     25.0     4005214.5          17.7
PV filter         25.0     4005214.5          17.7
>=2 LJs            0.0           0.0           0.0
2mu2e              0.0           0.0           0.0


QCD_Pt20To30
cut name         raw N    weighted N    weighted %
-------------  -------  ------------  ------------
None             634.0    66215056.0         100.0
pass triggers     45.0     4699806.5           7.1
PV filter         45.0     4699806.5           7.1
>=2 LJs            0.0           0.0           0.0
2mu2e              0.0           0.0           0.0


QCD_Pt30To50
cut name         raw N    weighted N    weighted %
-------------  -------  ------------  ------------
None            3159.0   144321296.0         100.0
pass triggers    155.0     7081292.0           4.9
PV filter        155.0     7081292.0   

In [5]:
#Variables 
# Hist_2mu = ["mulj_egmlj_invmass", "lj_lj_absdR", "lj_lj_absdeta"]


#channels: cuts to be applied (slections.yaml) 
channels = ["baseNoLj", "bkg_study_isopdisp", "bkg_study_isopdisp_2lj", 
            "bkg_study_isopdisp_2lj_dPhi_4mu", 
            
            "bkg_study_isopdisp_2lj_dPhi_2mu2e", 
            "bkg_study_isopdisp_2lj_dPhi2p2_2mu2e",
            "bkg_study_isopdisp_2lj_dPhi1p8_2mu2e"
           ]

ch1 = channels[0] #base
ch2 = channels[1] # iso + disp
ch3 = channels[2] # 2lj
ch4 = channels[3] # dphi2 4mu
ch5 = channels[4] # dphi2 2mu
ch6 = channels[5] # dphi2.2 2mu
ch7 = channels[6] # dphi2.2 2mu

all_cha = [ch1, ch2, ch3, ch4, ch5, ch6, ch7]
all_cha_names = ["base", "Iso + Disp", ">2LJ", "4mu", "2mu2e"]
col = ["r", "g", "b"]

In [4]:
# Signal and BKG files

mxx_2mu = ["2Mu2E_200GeV_5p0GeV_100p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm", "2Mu2E_800GeV_5p0GeV_50p0mm","2Mu2E_1000GeV_5p0GeV_40p0mm",]# Mxx_5_300
mzd_2mu = ["2Mu2E_500GeV_0p25GeV_4p0mm", "2Mu2E_500GeV_1p2GeV_19p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm", ] # 500_Mzd_300
lxy_2mu = ["2Mu2E_500GeV_1p2GeV_0p019mm", "2Mu2E_500GeV_1p2GeV_0p19mm", "2Mu2E_500GeV_1p2GeV_1p9mm", "2Mu2E_500GeV_1p2GeV_9p6mm", "2Mu2E_500GeV_1p2GeV_19p0mm"] # 500_1.2_Lxy

mxx_4mu = [    "4Mu_200GeV_5p0GeV_200p0mm",    "4Mu_500GeV_5p0GeV_80p0mm",    "4Mu_800GeV_5p0GeV_50p0mm",    "4Mu_1000GeV_5p0GeV_40p0mm",]
mzd_4mu = [    "4Mu_500GeV_0p25GeV_0p004mm",    "4Mu_500GeV_1p2GeV_19p0mm", "4Mu_500GeV_5p0GeV_80p0mm",]
lxy_4mu = [    "4Mu_500GeV_1p2GeV_0p019mm",     "4Mu_500GeV_1p2GeV_0p19mm",    "4Mu_500GeV_1p2GeV_1p9mm",    "4Mu_500GeV_1p2GeV_9p6mm",    "4Mu_500GeV_1p2GeV_19p0mm",]

sam_4mu = mxx_4mu + mzd_4mu + lxy_4mu

bkgdyj = ["DYJetsToMuMu_M10to50",    "DYJetsToMuMu_M50",]
bkgttj = ["TTJets"]
bkgqcd = ["QCD_Pt15To20",    "QCD_Pt20To30",    "QCD_Pt30To50",    "QCD_Pt50To80",    "QCD_Pt80To120",]
bkgocd = ["QCD_Pt120To170",    "QCD_Pt170To300",    "QCD_Pt300To470",    "QCD_Pt470To600",    "QCD_Pt600To800", "QCD_Pt1000",] # "QCD_Pt800To1000", # does not work
allqcd = bkgqcd + bkgocd

In [ ]:
raise SystemExit("Notebook stopped intentionally after this cell.")